# Finding Demand Drivers with PROC GLMSELECT: Stepwise, LASSO, and Validated Forward Selection

## Executive Summary

A retail analytics team uses **PROC GLMSELECT** to discover which marketing and pricing levers actually move weekly unit sales, separating true demand drivers from noise. Stepwise selection scored by SBC, LASSO with 5-fold cross validation, and a holdout-validated forward search all recover the **same set of true drivers** — own price, competitor price, ad spend, email volume, promotions, holidays, the Northeast region, and the Online channel — and every one of the four planted decoys (`temp_f`, `noise1`, `noise2`, `noise3`) is automatically discarded.

The methods also agree closely on the magnitudes: each estimates an own-price effect near **-28 units per dollar** and a competitor-price effect near **+14**, the substitution signs the data-generating equation built in. The only disagreement is at the margin — the cross-validated LASSO additionally keeps a small, statistically negligible `Region=Midwest` contrast (estimate -8.3 with a standard error of 8.3) that stepwise and forward selection both drop. A driver list that survives three different selection philosophies is far more trustworthy than one tuned to a single rule.

## Data Sources

All data in this notebook is **synthetic** and generated inline (no external files, seed `20260531`). It mimics one season of store-week demand panels for a consumer-goods retailer.

| Dataset | Rows | Grain | Key variables |
|---------|------|-------|---------------|
| `demand` | 100 | Store-week | `units` (response: weekly units sold); `price`, `competitor` (own & rival shelf price); `adspend`, `email` (media pressure); `promo`, `holiday` (event flags); `region`, `channel` (CLASS effects); `temp_f`, `noise1`-`noise3` (decoy/irrelevant predictors) |

`units` is built from a known linear equation so that the "correct" set of drivers is verifiable; `temp_f` and the three `noise` columns carry no true signal and exist to test whether each selection method discards them.

# Finding Demand Drivers with PROC GLMSELECT

A category manager has dozens of candidate explanators for weekly sales: own price, competitor price, advertising spend, email volume, promotions, holidays, store region, sales channel, even the weather. Throwing them all into one regression invites overfitting and unstable coefficients. **PROC GLMSELECT** automates the search for a parsimonious model, supporting stepwise, LASSO, elastic net, and least-angle selection with built-in cross validation and holdout partitioning.

In this notebook we:

1. Generate a realistic synthetic store-week demand panel with a *known* set of true drivers (plus deliberate decoy variables).
2. Run **stepwise selection** scored by SBC.
3. Run **LASSO** with 5-fold cross validation.
4. Run **forward selection validated on a 30% holdout**.

A good selection method should recover the real drivers and discard the noise — let's see.

## 1. Generate a synthetic demand panel

The response `units` is constructed from an explicit linear equation, so we know the ground truth: price and competitor price, ad spend, email volume, the promo and holiday flags, plus region and channel effects all matter. The variables `temp_f`, `noise1`, `noise2`, and `noise3` are pure decoys with no real relationship to sales. A `call streaminit` seed makes the data reproducible.

In [1]:
/* ---------------------------------------------------------------
   Synthetic store-week demand panel for a consumer-goods retailer.
   'units' follows a KNOWN equation; temp_f and noise1-3 are decoys.
   --------------------------------------------------------------- */
data demand;
    call streaminit(20260531);
    length region $9 channel $8 promo $3;
    do store_week = 1 to 100;
        /* Region mix */
        u1 = rand('uniform');
        if u1 < 0.34 then region = 'Northeast';
        else if u1 < 0.67 then region = 'Midwest';
        else region = 'South';

        /* Sales channel */
        if rand('uniform') < 0.45 then channel = 'Store';
        else channel = 'Online';

        /* Pricing and media drivers */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Event flags and an irrelevant weather decoy */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        if rand('uniform') < 0.40 then promo = 'Yes';
        else promo = 'No';

        /* Pure noise predictors (no true effect) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Weekly units sold from a known structural equation */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        if units < 0 then units = 0;
        output;
    end;
    keep region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
run;

NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## 2. Profile the data

Before modeling, confirm the response and the main continuous candidates are on sensible scales.

In [2]:
proc means data=demand n mean std min max maxdec=1;
    var units price competitor adspend email;
    title 'Weekly demand and candidate drivers';
run;

                                          Weekly demand and candidate drivers                                           

                                                  The MEANS Procedure

 Variable           N        Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 units            100       874.8       136.3       508.6      1162.3
 price            100        14.0         3.4         8.0        19.9
 competitor       100        13.8         3.4         8.1        19.9
 adspend          100       990.6       726.9        50.0      3358.0
 email            100        45.5        26.4         0.0        99.0
 --------------------------------------------------------------------



NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Stepwise selection scored by SBC

Stepwise selection alternately adds and removes effects, here judged by the **Schwarz Bayesian Criterion (SBC)** for both the entry test (`select=sbc`) and the final model choice (`choose=sbc`). SBC penalizes complexity more heavily than AIC, favoring leaner models.

Key statements and options:

- `CLASS region channel promo / param=reference` declares the categorical predictors with reference-cell coding.
- `selection=stepwise(select=sbc choose=sbc)` drives the search.
- `details=summary` prints the step-by-step selection summary; `stb` adds standardized coefficients so effects on different scales are comparable.
- `output out=demand_scored p=predicted r=residual` saves fitted values and residuals for downstream scoring.

Read the **Stepwise Selection Summary** as the search trace: it starts from the full 12-effect model and *removes* effects one at a time, dropping `noise1`, `noise2`, `temp_f`, the `Region=Midwest` contrast, and `noise3` in turn, because each removal lowers SBC. What survives in the **Parameter Estimates** table is the chosen model.

In [3]:
proc glmselect data=demand seed=20260531;
    class region channel promo / param=reference;
    model units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(select=sbc choose=sbc)
          details=summary stb;
    output out=demand_scored p=predicted r=residual;
    title 'Stepwise selection of demand drivers (SBC)';
run;

                                          Weekly demand and candidate drivers                                           

The GLMSELECT Procedure


Dependent Variable: UNITS


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                Stepwise Selection Summary                                                 

    Step    Action          Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  --------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed          NOISE1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136
       2   Removed         

NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 4. LASSO with cross validation

The LASSO shrinks coefficients toward zero, performing selection and regularization simultaneously. Rather than stopping at a fixed criterion, we let **5-fold cross validation** pick the point on the LASSO path with the best out-of-sample error.

- `selection=lasso(choose=cv stop=none)` traces the full LASSO path and selects the CV-optimal step.
- `cvmethod=random(5)` assigns observations to 5 random folds.

The **LASSO Selection Summary** shows the order in which effects enter as the penalty relaxes: `price` first, then `adspend`, `competitor`, the Northeast region, `email`, `promo`, and `holiday` — all seven true signals enter before any decoy. Cross validation then chooses the step where out-of-sample PRESS is lowest, and the resulting **Parameter Estimates** table keeps exactly the genuine drivers (plus a small `Region=Midwest` term) while excluding `temp_f`, `noise1`, `noise2`, and `noise3`, which only enter at the very end of the path.

In [4]:
proc glmselect data=demand seed=20260531;
    class region channel promo / param=reference;
    model units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv stop=none)
          cvmethod=random(5);
    title 'LASSO with 5-fold cross validation';
run;

                                          Weekly demand and candidate drivers                                           

The GLMSELECT Procedure


Dependent Variable: UNITS


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                         LASSO Selection Summary                                                          

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ----------------  -----------------  --------  ------

NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Forward selection validated on a holdout

A third, complementary strategy: build the model by **forward selection** (effects only enter, never leave), but stop where performance on an independent holdout sample is best — guarding directly against overfitting.

- `partition fraction(validate=0.30)` randomly reserves 30% of rows for validation, leaving 70 training and 30 validation observations.
- `selection=forward(select=aic choose=validate)` adds effects by AIC but chooses the final model by validation-sample error.

The **Partition Fractions** table confirms the 70/30 split. Forward selection then enters effects until the validation error stops improving; the eight effects in the final **Parameter Estimates** table are precisely the true drivers, with the four decoys never admitted. When three methods built on different principles land on the same drivers, the model is far more likely to be real than an artifact of any single selection rule.

In [5]:
proc glmselect data=demand seed=20260531;
    class region channel promo / param=reference;
    model units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(select=aic choose=validate);
    partition fraction(validate=0.30);
    title 'Forward selection validated on a 30% holdout';
run;

                                          Weekly demand and candidate drivers                                           

The GLMSELECT Procedure


Dependent Variable: UNITS


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                        Forward Selection Summary                                                         

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ----------------  ----------------- 

NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Interpreting the results

All three selection strategies recover the **same set of true demand drivers** and discard every decoy. Reading straight from the **Parameter Estimates** tables:

- **Price** is the dominant driver. Its standardized coefficient in the stepwise model is **-0.70**, by far the largest in magnitude, and the raw slope lands between **-27.8** (stepwise and LASSO) and **-28.6** (forward) units per dollar — almost exactly the -28 the data was generated with. **Competitor price** moves demand the other way at roughly **+14.4** across all three fits, the substitution behavior a category manager expects.
- **Ad spend** (about +0.062 units per dollar) and **email volume** (about +1.2 units per send) both lift sales, quantifying media response.
- **Promotions** and **holidays** carry the largest event effects: the `Promo=No` contrast runs about **-51 to -57** relative to a promoted week, and the holiday lift is roughly **+66 to +76** units. The **Northeast region** (about +49 to +55) and **Online channel** (about +24 to +32) carry positive baseline effects.
- Critically, every decoy — `temp_f`, `noise1`, `noise2`, `noise3` — is **dropped** by stepwise and forward selection and is excluded from the CV-chosen LASSO model. Each method recovered the structural signal and ignored the noise.

The three methods are not byte-for-byte identical: stepwise (SBC) and the holdout-validated forward search settle on the same eight effects, while the cross-validated LASSO additionally retains a small `Region=Midwest` contrast (-8.3, standard error 8.3) — a difference at the noise floor rather than a substantive disagreement. That a driver list survives stepwise SBC, cross-validated LASSO, and holdout-validated forward selection is the real takeaway: a finding robust to three different selection philosophies is far more credible than one tuned to a single criterion. With `OUTPUT OUT=demand_scored` capturing fitted values and residuals, the same workflow extends naturally to scoring next quarter's planned price and promotion calendar.